# ELA + YOLOv8 Manipulation Detection — Colab runner

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/friskytzy/ela-yolo-pipeline/blob/main/notebooks/colab_pipeline.ipynb)

End-to-end runner for the ELA + YOLOv8 pipeline.

**Before running:**
1. `Runtime → Change runtime type → GPU` (T4 free is enough; A100/L4 if you have Pro).
2. Open the **🔑 Secrets** sidebar (key icon on the left) and add a secret named `ROBOFLOW_API_KEY`. Toggle *Notebook access* on. Get the key from <https://app.roboflow.com/settings/api>.
3. `Runtime → Run all`.

Total time on T4: ~10–15 minutes for 100 images / 50 epochs. Re-run individual cells to iterate without reloading data.

## 1. Verify GPU

In [ ]:
!nvidia-smi -L || echo 'No GPU detected. Runtime > Change runtime type > GPU.'

## 2. Clone the repo and install dependencies

We pin to the `main` branch. The whole repo is small (figures + ~700 LOC of Python). Dependencies install in ~1 min on Colab.

In [ ]:
import os, pathlib, subprocess, sys

REPO_URL = 'https://github.com/friskytzy/ela-yolo-pipeline.git'
REPO_DIR = '/content/ela-yolo-pipeline'

if not pathlib.Path(REPO_DIR).exists():
    subprocess.check_call(['git', 'clone', '--depth=1', REPO_URL, REPO_DIR])
else:
    subprocess.check_call(['git', '-C', REPO_DIR, 'pull', '--ff-only'])

os.chdir(REPO_DIR)
print('Working dir:', os.getcwd())

In [ ]:
!pip install -q -r requirements.txt

## 3. Provide the Roboflow API key

Preferred: store it once as a Colab Secret named `ROBOFLOW_API_KEY` and toggle *Notebook access* on. Fallback: a hidden `getpass` prompt if the secret is unavailable. The value is **never** printed and **never** committed.

In [ ]:
import os

key = os.environ.get('ROBOFLOW_API_KEY')
if not key:
    try:
        from google.colab import userdata
        key = userdata.get('ROBOFLOW_API_KEY')
    except Exception:
        key = None
if not key:
    import getpass
    key = getpass.getpass('Roboflow API key (input hidden): ').strip()

assert key, 'No Roboflow API key provided.'
os.environ['ROBOFLOW_API_KEY'] = key
print('ROBOFLOW_API_KEY set ({} chars).'.format(len(key)))

## 4. Run the full pipeline

All knobs are editable below. Defaults match the pilot used in `reports/RESULTS.md` (100 authentic, 50 epochs, imgsz 640). Tweak as desired:

* T4 free: `NUM_IMAGES=300`, `EPOCHS=80`, `IMG_SIZE=640`, `BATCH=16` (≈15 min).
* L4 / A100 (Pro): `NUM_IMAGES=1000`, `EPOCHS=100`, `IMG_SIZE=1024`, `BATCH=16` (≈30 min).

`--device 0` selects the first GPU automatically; pass `--device cpu` to force CPU.

In [ ]:
NUM_IMAGES = 100
EPOCHS     = 50
IMG_SIZE   = 640
BATCH      = 8
DEVICE     = '0'        # '0' = first GPU, 'cpu' = CPU

cmd = (
    f'python scripts/run_pipeline.py '
    f'--num-images {NUM_IMAGES} '
    f'--epochs {EPOCHS} '
    f'--imgsz {IMG_SIZE} '
    f'--batch {BATCH} '
    f'--device {DEVICE}'
)
print(cmd)
import subprocess, sys
subprocess.check_call(cmd, shell=True)

## 5. Inspect the analysis metrics

`reports/metrics.json` contains PSNR/SSIM, the inside-vs-outside bbox ELA-error ratio for each tampering kind, and the YOLO-ready dataset stats.

In [ ]:
import json, pathlib

metrics = json.loads(pathlib.Path('reports/metrics.json').read_text())

print(f"n_authentic = {metrics['n_authentic']}, n_tampered = {metrics['n_tampered']}")
print(f"PSNR(orig vs JPEG-90) = {metrics['psnr_orig_vs_recompressed']['mean']:.2f} dB")
print(f"SSIM(orig vs JPEG-90) = {metrics['ssim_orig_vs_recompressed']['mean']:.4f}")
print()
print('inside-vs-outside bbox (overall):')
for k, v in metrics['inside_vs_outside_bbox'].items():
    print(f'  {k:>40s}: {v}')
print()
for kind, vals in metrics.get('per_kind', {}).items():
    print(f'inside-vs-outside ({kind}):')
    for k, v in vals.items():
        print(f'  {k:>40s}: {v}')
    print()

## 6. Visualise ELA error distributions and heatmaps

In [ ]:
from IPython.display import Image, display, Markdown
import pathlib

for label, path in [
    ('Per-image ELA error histogram (per kind)', 'reports/figures/ela_error_per_kind.png'),
    ('Inside-vs-outside bbox ELA error scatter', 'reports/figures/inside_vs_outside.png'),
    ('ELA heatmap examples (copy-move + removal)', 'reports/figures/ela_heatmap_examples.png'),
]:
    display(Markdown(f'### {label}'))
    display(Image(filename=path))

## 7. YOLOv8 training summary

`run_pipeline.py` writes Ultralytics outputs to `runs/detect/runs/ela_yolo/train/` (training plots, PR / F1 curves, confusion matrix, prediction grids). Final metrics are echoed by the train stage; we re-derive them from `results.csv` for completeness.

In [ ]:
import pandas as pd, pathlib

csv = pathlib.Path('runs/detect/runs/ela_yolo/train/results.csv')
if csv.exists():
    df = pd.read_csv(csv)
    last = df.iloc[-1]
    print('Final epoch metrics:')
    print(f"  precision   = {last['metrics/precision(B)']:.4f}")
    print(f"  recall      = {last['metrics/recall(B)']:.4f}")
    print(f"  mAP@0.5     = {last['metrics/mAP50(B)']:.4f}")
    print(f"  mAP@0.5:.95 = {last['metrics/mAP50-95(B)']:.4f}")
else:
    print('results.csv not found — did the train stage run?')

In [ ]:
from IPython.display import Image, display, Markdown
import pathlib

train_dir = pathlib.Path('runs/detect/runs/ela_yolo/train')
for label, path in [
    ('Training curves', train_dir / 'results.png'),
    ('Precision-Recall curve', train_dir / 'BoxPR_curve.png'),
    ('Validation predictions (batch 0)', train_dir / 'val_batch0_pred.jpg'),
    ('Validation predictions (batch 1)', train_dir / 'val_batch1_pred.jpg'),
]:
    if path.exists():
        display(Markdown(f'### {label}'))
        display(Image(filename=str(path)))
    else:
        print(f'(skipped) {path} not found')

## 8. (Optional) Download artefacts back to your machine

Bundles the report figures + best YOLO weights into a single zip and triggers a browser download.

In [ ]:
import pathlib, zipfile

out = pathlib.Path('/content/ela_yolo_results.zip')
if out.exists():
    out.unlink()

with zipfile.ZipFile(out, 'w', zipfile.ZIP_DEFLATED) as zf:
    for src in pathlib.Path('reports').rglob('*'):
        if src.is_file():
            zf.write(src, arcname=src.relative_to('.'))
    best = pathlib.Path('runs/detect/runs/ela_yolo/train/weights/best.pt')
    if best.exists():
        zf.write(best, arcname=best.relative_to('.'))

print(f'Wrote {out} ({out.stat().st_size / 1024**2:.2f} MiB).')
try:
    from google.colab import files
    files.download(str(out))
except Exception as e:
    print('Auto-download unavailable:', e)
    print('Right-click the file in the Files sidebar to save it.')

---

**Tip — persistent storage**: Colab erases `/content` when the runtime stops. To keep the dataset/runs across sessions, mount Drive *before* cloning the repo:

```python
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive
# then clone there instead of /content
```